In [0]:
# Demo dataset from Databricks
display(dbutils.fs.ls("/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/"))

In [0]:
display(dbutils.fs.ls("/data/input/nyctaxi/parquet/"))

path,name,size,modificationTime
dbfs:/data/input/nyctaxi/parquet/part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet,part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet,374549044,1720944538000
dbfs:/data/input/nyctaxi/parquet/part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet,part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet,364876497,1720944572000


In [0]:
# Check the count and verify data scanning
# data copied at location - "/data/input/nyctaxi/parquet/"
df_parquet = spark.read.parquet("/data/input/nyctaxi/parquet/")
display(df_parquet)

vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,rate_code_id,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount
CMT,2010-02-12T08:13:37.000+0000,2010-02-12T08:23:22.000+0000,1,1.0,-73.993828,40.732739,1,0,-74.008402,40.733428,CRE,6.5,0.0,0.5,1.75,0.0,8.75
CMT,2010-02-12T20:48:01.000+0000,2010-02-12T21:01:40.000+0000,1,2.3,-74.007883,40.716298,1,0,-73.996248,40.744228,CRE,9.3,0.5,0.5,2.06,0.0,12.36
CMT,2010-02-12T16:25:23.000+0000,2010-02-12T16:47:31.000+0000,2,4.2,-74.001943,40.706967,1,1,-73.984553,40.671861,CAS,13.7,1.0,0.5,0.0,0.0,15.2
CMT,2010-02-13T02:02:16.000+0000,2010-02-13T02:06:49.000+0000,1,0.7,-74.003708,40.729451,1,0,-74.009183,40.735422,CRE,4.5,0.5,0.5,1.1,0.0,6.6
CMT,2010-02-12T20:49:29.000+0000,2010-02-12T21:04:18.000+0000,1,5.8,-73.993709,40.681982,1,0,-73.951812,40.723367,CRE,15.3,0.5,0.5,2.44,0.0,18.74
CMT,2010-02-13T07:51:56.000+0000,2010-02-13T07:56:42.000+0000,1,1.8,-74.000636,40.727351,1,0,-73.991558,40.748782,CRE,6.1,0.0,0.5,1.0,0.0,7.6
CMT,2010-02-12T11:12:37.000+0000,2010-02-12T11:22:08.000+0000,1,1.8,-74.004539,40.707313,1,0,-73.994224,40.703248,CAS,7.7,0.0,0.5,0.0,0.0,8.2
CMT,2010-02-13T06:59:54.000+0000,2010-02-13T07:03:14.000+0000,1,1.1,-74.012405,40.709634,1,0,-74.007683,40.724895,CRE,4.9,0.0,0.5,1.0,0.0,6.4
CMT,2010-02-12T04:20:26.000+0000,2010-02-12T04:24:21.000+0000,1,1.2,-74.005623,40.726305,1,0,-74.014805,40.716356,CRE,4.9,0.5,0.5,1.0,0.0,6.9
CMT,2010-02-12T06:52:51.000+0000,2010-02-12T06:57:34.000+0000,1,1.1,-74.008553,40.719237,1,0,-74.011661,40.707914,CAS,4.9,0.0,0.5,0.0,0.0,5.4


In [0]:
# Check filter data
# select count(1) from nyctaxi where vendor_id = 'VTS' and trip_distance > 1.8

df_parquet.where("vendor_id = 'VTS' and trip_distance > 1.8").count()

Out[8]: 5466328

In [0]:
# Write the data in partitioned format
df_parquet.write.format("parquet").mode("overwrite").partitionBy("vendor_id").option("path", "/data/input/nyctaxi/partitioned/").saveAsTable("nyctaxi_partitioned")

In [0]:
display(dbutils.fs.ls("/data/input/nyctaxi/partitioned/"))

path,name,size,modificationTime
dbfs:/data/input/nyctaxi/partitioned/_SUCCESS,_SUCCESS,0,1720952782000
dbfs:/data/input/nyctaxi/partitioned/_started_1872870739494801713,_started_1872870739494801713,0,1720952067000
dbfs:/data/input/nyctaxi/partitioned/vendor_id=CMT/,vendor_id=CMT/,0,0
dbfs:/data/input/nyctaxi/partitioned/vendor_id=DDS/,vendor_id=DDS/,0,0
dbfs:/data/input/nyctaxi/partitioned/vendor_id=VTS/,vendor_id=VTS/,0,0


In [0]:
df_partitioned = spark.read.parquet("/data/input/nyctaxi/partitioned/")

df_partitioned.where("vendor_id = 'VTS' and trip_distance > 1.8").count()

Out[14]: 5466328

In [0]:
%sql

select count(1) from nyctaxi_partitioned where vendor_id = 'VTS' and trip_distance > 1.8

count(1)
5466328
